In [1]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy.wcs import WCS
from reproject import reproject_to_healpix

In [2]:
def read_in_file(file_in):

    hdu  = fits.open(file_in)
    hdr  = hdu[0].header
    data = hdu[0].data
    
    return data, hdr

In [ ]:
dir_chime   = '/srv/data/chime/chime_IQUV_Mar2024_400_729/'
dir_dragons = '/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/'

Q_c, hdr_c = read_in_file(dir_chime + 'Q_400_729_Mar2024_new_gal_fix.fits')
U_c, hdr_c = read_in_file(dir_chime + 'U_400_729_Mar2024_new_gal_fix.fits')

Q_c_conv, hdr_c_conv = read_in_file(dir_chime + 'Q_400_729_Mar2026_gal_conv.fits')
U_c_conv, hdr_c_conv = read_in_file(dir_chime + 'U_400_729_Mar2026_gal_conv.fits')

Q_d, hdr_d = read_in_file(dir_dragons + 'dragons_freqK_gal_q.hpx.fits')
U_d, hdr_d = read_in_file(dir_dragons + 'dragons_freqK_gal_u.hpx.fits')

Q_d_conv, hdr_d_conv = read_in_file(dir_dragons + 'dragons_freqK_gal_conv_q.hpx.fits')
U_d_conv, hdr_d_conv = read_in_file(dir_dragons + 'dragons_freqK_gal_conv_u.hpx.fits')

In [ ]:
print(Q_c.shape)
print(Q_c_conv.shape)
print(Q_d.shape)
print(Q_d_conv.shape)

In [ ]:
freqs_d = WCS(hdr_d).all_pix2world(0,range(Q_d.shape[0]),0)[1]
freqs_c = WCS(hdr_c_conv).all_pix2world(0,range(Q_c_conv.shape[0]),0)[1]

In [ ]:
def make_ratios(freq_pick, nside):

    idx_d = np.abs(freqs_d-freq_pick).argmin()
    idx_c = np.abs(freqs_c-freq_pick).argmin()
    print(f'Making channel: DRAGONS={freqs_d[idx_d]/1e6}, CHIME={freqs_c[idx_c]/1e6}')

    # DRAGONS map slices:
    Q_d_use = Q_d[idx_d]
    U_d_use = U_d[idx_d]
    Q_d_conv_use = Q_d_conv[idx_d]
    U_d_conv_use = U_d_conv[idx_d]

    # CHIME maps slices:
    Q_c_use, __ = reproject_to_healpix((Q_c[idx_c], WCS(hdr_c).dropaxis(2)), coord_system_out='G', nside=nside)
    U_c_use, __ = reproject_to_healpix((U_c[idx_c], WCS(hdr_c).dropaxis(2)), coord_system_out='G', nside=nside)
    Q_c_conv_use = hp.ud_grade(Q_c_conv[idx_c], nside)
    U_c_conv_use = hp.ud_grade(U_c_conv[idx_c], nside)

    Q_c_conv_use[Q_c_conv_use==hp.UNSEEN] = np.nan
    U_c_conv_use[U_c_conv_use==hp.UNSEEN] = np.nan

    ratio_Q = Q_d_use/Q_c_use 
    ratio_U = U_d_use/U_c_use 
    ratio_Q_conv = Q_d_conv_use/Q_c_conv_use 
    ratio_U_conv = U_d_conv_use/U_c_conv_use 

    #hp.mollview(Q_c_conv_use)

    return ratio_Q, ratio_U, ratio_Q_conv, ratio_U_conv

In [ ]:
nside = hp.npix2nside(Q_d_conv.shape[1])
print(nside)

ratio_Q, ratio_U, ratio_Q_conv, ratio_U_conv = make_ratios(716e6, nside)
hp.mollview(ratio_Q, min=-5,max=5, cmap='RdBu_r')
hp.mollview(ratio_U, min=-5,max=5, cmap='RdBu_r')
hp.mollview(ratio_Q_conv, min=-5,max=5, cmap='RdBu_r')
hp.mollview(ratio_U_conv, min=-5,max=5, cmap='RdBu_r')

In [ ]:
plt.hist(ratio_Q, bins=101, range=(-5,5), alpha=0.5);
plt.hist(ratio_U, bins=101, range=(-5,5), alpha=0.5);

In [ ]:
plt.hist(ratio_Q_conv, bins=101, range=(-5,5), alpha=0.5);
plt.hist(ratio_U_conv, bins=101, range=(-5,5), alpha=0.5);